# Preprocessing DMFT Output Files → SVO Dataset

This notebook reads raw CT-HYB DMFT simulation output files (HDF5 format) produced
by TRIQS/DFTTools, extracts the Matsubara self-energy Σ(iω) and non-interacting
Green's function G₀(iω), and saves a clean compressed numpy array ready for training.

**Input:**  `sims_SVO/` directory containing HDF5 files named `svo_beta40_nwarmup<N>.h5`  
**Output:** `data/dataSVO.npz`  

> **Note:** The raw `.h5` simulation files are not included in this repository due to their
> size (~22 MB each). The already-preprocessed `data/dataSVO.npz` is provided so you can
> skip directly to notebook `02_train_svo_lstm.ipynb`. Run this notebook only if you have
> the raw HDF5 files.

## 0. Imports

In [ ]:
import matplotlib.pyplot as plt
import h5py
import numpy as np
import glob

## 1. Define HDF5 Reader

Each file contains 25 DMFT self-consistency iterations and 6 spin-orbital channels
(t₂g: down_0/1/2, up_0/1/2). We extract Im Σ(iω) and Im G₀(iω) from every
iteration × channel pair, masking the frequency grid to the positive window (0, 10) eV
which gives 64 uniformly-spaced Matsubara points.

`readfile_special` reads only the last iteration — useful for files where early
iterations are noisy (e.g. LVO).

In [ ]:
def readfile(filename):
    
    
    with h5py.File(filename, 'r') as ar:
        name = filename.split('/')[2]

        iteration_number = 25 # total n iterations
        tot = iteration_number*6
        Sigma_iw=ar['dmft_output']['Sigma_iw']
        freq=np.linspace(-161,161,2050)
        mask = (freq > 0) & (freq < 10)
        freq_sel = freq[mask]


        nwarmupstep = int(name.split('.')[0].split('_')[2][7:])
        list_nwarmup = np.repeat(nwarmupstep,tot)
        real_all_sigma = np.zeros([tot,len(freq_sel)])
        img_all_sigma = np.zeros([tot,len(freq_sel)])

        real_all_g0 = np.zeros([tot,len(freq_sel)])
        img_all_g0 = np.zeros([tot,len(freq_sel)])
    
    
        count =0
        spin_list =['down_0','down_1','down_2','up_0','up_1','up_2']
        for it in range(1,iteration_number+1):
            
            chunk = ar['dmft_output']['Sigma-%s'%(it)]
            chunk2 = ar['dmft_output']['G0-%s'%(it)]
            for spin in spin_list:
                freq = Sigma_iw[spin]['mesh']['domain']
                data1 = np.asarray(chunk[spin]['data'][()]).reshape(2050,2)
                data_real_sigma,data_img_sigma = data1[:,0], data1[:,1]
                data_real_sigma_sel = data_real_sigma[mask]
        
                data_img_sigma_sel = data_img_sigma[mask]
                real_all_sigma[count] = data_real_sigma_sel
                img_all_sigma[count] = data_img_sigma_sel


                data2 = np.asarray(chunk2[spin]['data'][()]).reshape(2050,2)
                data_real_g0,data_img_g0 = data2[:,0], data2[:,1]
                data_real_g0_sel = data_real_g0[mask]
                data_img_g0_sel = data_img_g0[mask]
                real_all_g0[count] = data_real_g0_sel
                img_all_g0[count] = data_img_g0_sel



                
                count = count+1
        return list_nwarmup, real_all_sigma,img_all_sigma,real_all_g0,img_all_g0
  

In [ ]:
#### for lvo, read the last iteration to get rid of noisy files

def readfile_special(filename):
    
    
    with h5py.File(filename, 'r') as ar:
        name = filename.split('/')[2]

        iteration_number = 3 #25 # total n iterations
        tot = 6
        Sigma_iw=ar['dmft_output']['Sigma_iw']
        freq=np.linspace(-161,161,2050)
        mask = (freq > 0) & (freq < 10)
        freq_sel = freq[mask]


        nwarmupstep = int(name.split('.')[0].split('_')[2][7:])
        list_nwarmup = np.repeat(nwarmupstep,tot)
        real_all_sigma = np.zeros([tot,len(freq_sel)])
        img_all_sigma = np.zeros([tot,len(freq_sel)])

        real_all_g0 = np.zeros([tot,len(freq_sel)])
        img_all_g0 = np.zeros([tot,len(freq_sel)])
    
    
        count =0
        spin_list =['down_0','down_1','down_2','up_0','up_1','up_2']
        
        chunk = ar['dmft_output']['Sigma-%s'%(iteration_number)]
        chunk2 = ar['dmft_output']['G0-%s'%(iteration_number)]
        for spin in spin_list:
            freq = Sigma_iw[spin]['mesh']['domain']
            data1 = np.asarray(chunk[spin]['data'][()]).reshape(2050,2)
            data_real_sigma,data_img_sigma = data1[:,0], data1[:,1]
            data_real_sigma_sel = data_real_sigma[mask]
            data_img_sigma_sel = data_img_sigma[mask]
            real_all_sigma[count] = data_real_sigma_sel
            img_all_sigma[count] = data_img_sigma_sel


            data2 = np.asarray(chunk2[spin]['data'][()]).reshape(2050,2)
            data_real_g0,data_img_g0 = data2[:,0], data2[:,1]
            data_real_g0_sel = data_real_g0[mask]
            data_img_g0_sel = data_img_g0[mask]
            real_all_g0[count] = data_real_g0_sel
            img_all_g0[count] = data_img_g0_sel



                
            count = count+1
        return list_nwarmup, real_all_sigma,img_all_sigma,real_all_g0,img_all_g0
  

## 2. Load and Accumulate All Simulation Files

Glob all HDF5 files in `sims_SVO/`, call `readfile()` on each, and
accumulate the arrays. Files that fail to parse are skipped with a warning.

In [ ]:


allfiles = glob.glob('./sims_SVO/*')
print(allfiles)
#allfiles = ['./sims_LVO/LVO_beta40_nwarmup350000.h5']#['./sims/svo_beta40_nwarmup620000.h5','./sims/svo_beta40_nwarmup640000.h5']
nwarmup_lists =[]
real_sigma_all =[]
img_sigma_all =[]

real_g0_all =[]
img_g0_all =[]
for mf in allfiles:
    try:
        nwarmup, real_sigma, img_sigma, real_g0,img_g0 = readfile(mf)
        
        for x,y,z,m,n in zip(real_sigma,img_sigma,nwarmup,real_g0,img_g0):
            real_sigma_all.append(x)
            img_sigma_all.append(y)
            nwarmup_lists.append(z)

            real_g0_all.append(m)
            img_g0_all.append(n)
        
        
    except:
        print('something wrong for file', mf)



## 3. Convert, Shuffle, and Save Dataset

Stack accumulated lists into numpy arrays, apply a shared random shuffle so
samples from all nwarmup values are interleaved, then save as a compressed `.npz`.

In [ ]:
nwarmup_lists = np.array(nwarmup_lists)
real_all_sigma = np.asarray(real_sigma_all)
img_all_sigma = np.asarray(img_sigma_all)
real_all_g0 = np.asarray(real_g0_all)
img_all_g0 = np.asarray(img_g0_all)
indices = np.arange(real_all_sigma.shape[0])
# Shuffle the indices
np.random.shuffle(indices)

# Use the shuffled indices to shuffle both arrays in the same order
real_shuff_sigma = real_all_sigma[indices]
img_shuff_sigma = img_all_sigma[indices]

real_shuff_g0 = real_all_g0[indices]
img_shuff_g0 = img_all_g0[indices]
nwarmup_shuff = nwarmup_lists[indices]
print(nwarmup_shuff)



In [ ]:
np.savez_compressed('data/dataSVO.npz',nwarupsteps=nwarmup_shuff, realpart_sigma=real_shuff_sigma,imgpart_sigma=img_shuff_sigma,realpart_g0=real_shuff_g0,imgpart_g0=img_shuff_g0)

## 4. Quick Visual Sanity Check

Plot a sample of Im Σ(iω) curves to confirm the data loaded correctly.

In [ ]:
for i in range(60):
    plt.plot(img_shuff_sigma[i],'b-')
plt.show()